# **FINE-TUNE & PUSHING TO HF**

##### This notebook contains code to fine-tune a BERT model on Consumer Financial Protection Bureau (CFPD) complaints narratives from customers. The fine-tuned model is then pushed to Hugging Face to be used by the public.

In [1]:
# gdrive setup
from google.colab import drive
from google.colab import userdata
drive.mount('/content/drive')


DATA_DIR = "/content/drive/MyDrive/ml_projects/data/complaints_data"

Mounted at /content/drive


In [2]:
from huggingface_hub import notebook_login
notebook_login(skip_if_logged_in=False)

In [3]:
# load necessary libraries

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import pandas as pd
import numpy as np
import re
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             precision_recall_fscore_support,classification_report)
from transformers import DataCollatorWithPadding
from transformers import Trainer, TrainingArguments
from tqdm import tqdm
import random
from transformers import pipeline
import torch
import os

In [4]:
def set_seeds(seed: int = 42):
    """Set seeds for reproducibility."""
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    eval("setattr(torch.backends.cudnn, 'deterministic', True)")
    eval("setattr(torch.backends.cudnn, 'benchmark', False)")
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds()

In [5]:
# load model + data

# load data
data_files = {"train":DATA_DIR+"/train.csv",
              "validation":DATA_DIR+"/val.csv"}
raw_dataset = load_dataset("csv", data_files=data_files)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [6]:
# an example of a complaint
raw_dataset['train']['complaint_what_happened'][1]

'I am writing to file a formal complaint regarding my federal student loans serviced by MOHELA. \nMy name is XXXX XXXX XXXX, address : XXXX XXXX XXXX XXXX, XXXX, IL XXXX. Phone : XXXX. Email : XXXX. \nI have been employed full-time since XX/XX/XXXX by XXXX  of XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX, a 5XXXXXXXX XXXX XXXX XXXX XXXX XXXX XXXX organization that provides emergency food and supplies to 37 food pantries in the poorest counties of XXXX  XXXX Kentucky XXXX \nKey Issues : All of my loans ( ~ {$88000.00} balance as of XX/XX/XXXX ) are currently in Administrative Forbearance that I never requested. \nMy employer was ruled ineligible for PSLF after XX/XX/XXXX in a XX/XX/XXXX Federal Student Aid letter, even though the IRS reinstated our XXXXXXXX XXXX XXXX XXXX XXXX XXXX XXXX  status on the same day ( XX/XX/XXXX ) with full retroactive effect. \nNo payments have counted toward PSLF since XX/XX/XXXX, despite continuous qualifying employment. \nMultiple loans I paid off in full

In [7]:
# create mapping of labels to idx (used as y-values) as well idx to labels (to convert predictions to label name(s))
label2idx = {label_name: idx for idx, label_name in enumerate(sorted(raw_dataset['train'].unique("labels")))}
print(label2idx)
idx2label = {idx:label_name for label_name, idx in label2idx.items()}

# apply label to index to labels column
raw_dataset_labelled = raw_dataset.map(lambda x: {"labels": label2idx[x['labels']]})

{'Banking Operations': 0, 'Cards & Payments': 1, 'Collections & Recovery': 2, 'Consumer Lending': 3, 'Credit Reporting & Disputes': 4, 'Money Transfer and Payments': 5, 'Mortgage & Home Lending': 6}


Map:   0%|          | 0/14198 [00:00<?, ? examples/s]

Map:   0%|          | 0/3550 [00:00<?, ? examples/s]

In [8]:
# function to clean sentences.

def clean_text(sentence):
  #sentence = sentence.lower()
  sentence = re.sub(r"([!\"'#$%&()*\+-/:;<=>?@\\\[\]^_`{|}~])", r" \1 ", sentence) # remove punctuations
  sentence = re.sub("[^A-Za-z0-9]+", " ", sentence) # remove non-alphanumeric characters
  # remove repeated XXXX (these are confidential data present in the dataset that were scrubbed)
  pattern = re.compile(r"(xx+\s*)+")
  #replace XX pattern with keyword: mask, to preseve some meaning/signal if present.
  sentence = pattern.sub("mask ", sentence)
  sentence = re.sub(" +", " ", sentence).strip() #remove repeated spaces
  sentence = re.sub(r"http\S+", "", sentence) # remove hyperlinks if present
  return sentence

#apply clean_text function to clean complaint sentences
raw_dataset_labelled = raw_dataset_labelled.map(lambda x:{"clean_text": [clean_text(text) for text in x['complaint_what_happened']]},
                                                batched=True)

Map:   0%|          | 0/14198 [00:00<?, ? examples/s]

Map:   0%|          | 0/3550 [00:00<?, ? examples/s]

In [9]:
# example of a cleaned complaint to remove
raw_dataset_labelled['train']['clean_text'][1]

'I am writing to file a formal complaint regarding my federal student loans serviced by MOHELA My name is XXXX XXXX XXXX address XXXX XXXX XXXX XXXX XXXX IL XXXX Phone XXXX Email XXXX I have been employed full time since XX XX XXXX by XXXX of XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX a 5XXXXXXXX XXXX XXXX XXXX XXXX XXXX XXXX organization that provides emergency food and supplies to 37 food pantries in the poorest counties of XXXX XXXX Kentucky XXXX Key Issues All of my loans 88000 00 balance as of XX XX XXXX are currently in Administrative Forbearance that I never requested My employer was ruled ineligible for PSLF after XX XX XXXX in a XX XX XXXX Federal Student Aid letter even though the IRS reinstated our XXXXXXXX XXXX XXXX XXXX XXXX XXXX XXXX status on the same day XX XX XXXX with full retroactive effect No payments have counted toward PSLF since XX XX XXXX despite continuous qualifying employment Multiple loans I paid off in full during XXXX are still showing incorrect balances

In [10]:
# the model to be fine-tuned

# distilbert uses BERT and it is fast.
checkpoint = "distilbert-base-uncased"

# load pretrained model
model = AutoModelForSequenceClassification.from_pretrained(checkpoint,
                                                           num_labels=7,
                                                           label2id=label2idx,
                                                           id2label=idx2label,
                                                           dtype='auto',
                                                           device_map='auto',
                                                           ignore_mismatched_sizes=True
                                                         )
# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [11]:
# create a tokenizer function to convert words in numbers to feed the models for training.

def tokenizer_function(example):
  return tokenizer(example['clean_text'], truncation=True, max_length=128) # avg text length is 126

# applying tokenizer function to extract numbers.
processed_dataset = raw_dataset_labelled.map(tokenizer_function, batched=True)


Map:   0%|          | 0/14198 [00:00<?, ? examples/s]

Map:   0%|          | 0/3550 [00:00<?, ? examples/s]

In [12]:
# create a custom function to feed into our Trainer to get evaluation metrics.
# metrics to used are macro metrics for precision, recall, f1 as well as weighted f1.

def compute_metrics(EvalPred):
  logits, labels = EvalPred

  preds = np.argmax(logits, axis=-1)
  metrics = precision_recall_fscore_support(y_true=labels, y_pred=preds, average='macro')

  all_metrics = {"accuracy": accuracy_score(y_true=labels, y_pred=preds),
                                    "precision": metrics[0],
                                    "recall": metrics[1],
                                    "f1": metrics[2],
                                    "weighted_f1": f1_score(y_true=labels, y_pred=preds, average='weighted')
                                    }
  return all_metrics

In [13]:
# a collator to help with padding sentences in a batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [14]:
# training arguments to fine-tune.
# the best model after training will be determined by the highest f1_macro score.

args = TrainingArguments("appcle/distilbert-base-uncased-cfpd",
                         bf16=True,
                         num_train_epochs=4,
                         per_device_train_batch_size=16,
                         save_strategy="epoch",
                         eval_strategy="epoch",
                         gradient_accumulation_steps=1,
                         load_best_model_at_end=True,
                         greater_is_better=True,
                         push_to_hub=True,
                         dataloader_num_workers=2,
                         metric_for_best_model='f1'
                         )

# trainer to fine-tune our model.
trainer = Trainer(model=model,
                  args=args,
                  data_collator=data_collator,
                  compute_metrics=compute_metrics,
                  train_dataset=processed_dataset['train'],
                  eval_dataset=processed_dataset['validation'],
                  processing_class=tokenizer
                  )

In [15]:
# perform fine-tuning.

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Weighted F1
1,0.914214,0.553906,0.816056,0.790967,0.793152,0.790680,0.815413
2,0.483807,0.526127,0.832676,0.815221,0.804689,0.809456,0.832834
3,0.302037,0.561913,0.830986,0.819438,0.803382,0.810915,0.830937
4,0.196955,0.634069,0.826761,0.807672,0.805467,0.806457,0.826870


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3552, training_loss=0.44901382278751684, metrics={'train_runtime': 1005.7044, 'train_samples_per_second': 56.47, 'train_steps_per_second': 3.532, 'total_flos': 1880939830622208.0, 'train_loss': 0.44901382278751684, 'epoch': 4.0})

### **PUSH TO HUB**

In [16]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/appcle/distilbert-base-uncased-cfpd/commit/9152544a2df1e30181871c986734345170faa1ff', commit_message='End of training', commit_description='', oid='9152544a2df1e30181871c986734345170faa1ff', pr_url=None, repo_url=RepoUrl('https://huggingface.co/appcle/distilbert-base-uncased-cfpd', endpoint='https://huggingface.co', repo_type='model', repo_id='appcle/distilbert-base-uncased-cfpd'), pr_revision=None, pr_num=None)

## **TESTING MODEL**

In [17]:
checkpoint = "appcle/distilbert-base-uncased-cfpd"

# pipe usage
pipe = pipeline("text-classification",
                model=checkpoint,
                device=-1,
                top_k=None)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [18]:
text = "I need to pay my debt but I do not like you keep heckling me"
pipe(text)

[[{'label': 'Collections & Recovery', 'score': 0.9633516669273376},
  {'label': 'Consumer Lending', 'score': 0.022568371146917343},
  {'label': 'Cards & Payments', 'score': 0.009196363389492035},
  {'label': 'Credit Reporting & Disputes', 'score': 0.003927602432668209},
  {'label': 'Banking Operations', 'score': 0.0005379121284931898},
  {'label': 'Mortgage & Home Lending', 'score': 0.0003162759239785373},
  {'label': 'Money Transfer and Payments', 'score': 0.00010183358972426504}]]

### **custom class to load model and perform batch predictions**

In [19]:
class BERTPredictor:
  def __init__(self, checkpoint:str, device:str='cpu'):
    self.model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
    self.device = torch.device(device)
    self.model = self.model.to(self.device)
    self.tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    self.model.eval()
    self.label2id = self.model.config.label2id

  def predict_batch(self, texts, batch_size=16):
    results = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Predicting batches...", colour="green"):
      batch = texts[i:i+batch_size]
      inputs = self.tokenizer(batch, padding=True, truncation=True, return_tensors='pt', max_length=128)
      inputs = {k:v.to(self.device) for k,v in inputs.items()}
      with torch.no_grad():
        outputs = self.model(**inputs)
      probabilities = torch.softmax(outputs.logits, dim=-1)
      predicted_index = torch.argmax(probabilities, dim=-1)


      for idx, pred in enumerate(predicted_index.tolist()):
        results.append({"label":self.model.config.id2label[pred], "score":probabilities[idx, pred].item()})
    return results

In [20]:
predictor = BERTPredictor(checkpoint, device='cuda')

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [21]:
val_complaint_list = raw_dataset_labelled['validation']['clean_text'][:]
# val predictions
batch_preds = predictor.predict_batch(val_complaint_list)
trf_preds = [predictor.label2id[score['label']] for score in batch_preds]
val_true = raw_dataset_labelled['validation']['labels'][:]

Predicting batches...: 100%|██████████| 222/222 [00:16<00:00, 13.14it/s]


In [22]:
accuracy = accuracy_score(y_true=val_true, y_pred=trf_preds)
weighted_f1 = f1_score(y_true=val_true, y_pred=trf_preds, average='weighted')
metrics = precision_recall_fscore_support(y_true=val_true, y_pred=trf_preds, average='macro')

print(f"{classification_report(y_true=val_true, y_pred=trf_preds)}\n")
metrics = {"accuracy": accuracy,
               "precision_macro": metrics[0],
               "recall_macro": metrics[1],
               "f1_macro":metrics[2],
               "weighted_f1":weighted_f1
               }

              precision    recall  f1-score   support

           0       0.79      0.84      0.81       754
           1       0.84      0.81      0.83       754
           2       0.87      0.89      0.88       962
           3       0.82      0.78      0.80       423
           4       0.75      0.70      0.73       110
           5       0.70      0.70      0.70       280
           6       0.96      0.90      0.92       267

    accuracy                           0.83      3550
   macro avg       0.82      0.80      0.81      3550
weighted avg       0.83      0.83      0.83      3550




In [23]:
# predict with proba.
# This produces the prediction as well as ordering probabilities
import time
def predict_with_proba_bert(text, predictor):
  start_time = time.time()
  text = clean_text(text)
  inputs = predictor.tokenizer(text, padding=True, max_length=128, truncation=True, return_tensors='pt')
  inputs = {k:v.to(predictor.device) for k,v in inputs.items()}
  with torch.no_grad():
    output = predictor.model(**inputs)
  probabilities = torch.softmax(output.logits, dim=-1)
  prediction = torch.argmax(probabilities, dim=-1)
  end_time = time.time()

  id2label = predictor.model.config.id2label
  label2id = list(predictor.model.config.label2id)
  all_probs = {label: prob for label, prob in zip(label2id, probabilities[0].tolist())}
  sorted_probs = dict(sorted(all_probs.items(), key=lambda x: x[1], reverse=True))
  return {"prediction":id2label[prediction.item()],
          "probabilities": sorted_probs,
          "latency": f"{(end_time - start_time) * 1000:.2f} ms"
          }
predictor = BERTPredictor(checkpoint, device='cpu')
predict_with_proba_bert("I need to pay my debt but I do not like you keep heckling me", predictor)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'prediction': 'Collections & Recovery',
 'probabilities': {'Collections & Recovery': 0.9633516669273376,
  'Consumer Lending': 0.022568373009562492,
  'Cards & Payments': 0.009196363389492035,
  'Credit Reporting & Disputes': 0.003927602432668209,
  'Banking Operations': 0.0005379121867008507,
  'Mortgage & Home Lending': 0.0003162759239785373,
  'Money Transfer and Payments': 0.00010183358972426504},
 'latency': '80.43 ms'}